# Environment Verification

This notebook validates that the DataGovKG development environment is correctly set up.

## Components tested

| Component | Purpose |
|-----------|---------|
| Python 3.11 | Programming language |
| `rdflib` | RDF graph manipulation |
| `SPARQLWrapper` | Communication with Fuseki via HTTP |
| `pyshacl` | SHACL validation engine |
| `openai` | LLM API client (for AI agents) |
| `streamlit` | Dashboard UI framework |
| Apache Jena Fuseki | Triplestore (running in Docker on port 3030) |

## Success criteria

- All Python libraries import without errors
- Fuseki responds to HTTP requests at `http://localhost:3030`
- A test RDF triple can be inserted and retrieved via SPARQL

If all cells run without errors, the environment is operational.

In [15]:
# Verify Python environment matches the project's conda environment
import sys
import platform

print(f"Python version  : {platform.python_version()}")
print(f"Python location : {sys.executable}")
print(f"Platform        : {platform.platform()}")

# Conda environments isolate dependencies; confirm we're in the right one
if "datagovkg" in sys.executable:
    print("\nRunning in 'datagovkg' environment")
else:
    print("\nWARNING: Not in 'datagovkg' environment")

Python version  : 3.11.15
Python location : C:\Users\bsula\anaconda3\envs\datagovkg\python.exe
Platform        : Windows-10-10.0.26200-SP0

Running in 'datagovkg' environment


In [16]:
# Core Libraries Version Check
# Uses importlib.metadata for a universal way to get versions
# (works for all libraries, including older ones like owlready2)

# Verify all required libraries are installed with expected versions
from importlib.metadata import version

core_libraries = [
    "rdflib",
    "pyshacl",
    "SPARQLWrapper",
    "owlready2",
    "openai",
    "streamlit",
    "pandas",
]

for lib in core_libraries:
    try:
        print(f"{lib:<15} : {version(lib)}")
    except Exception as e:
        print(f"{lib:<15} : MISSING ({e})")

rdflib          : 7.1.4
pyshacl         : 0.30.1
SPARQLWrapper   : 2.0.0
owlready2       : 0.48
openai          : 1.59.7
streamlit       : 1.41.1
pandas          : 2.2.3


In [17]:
# Fuseki Connectivity Test
# Purpose: Verify Fuseki is running and dataset is accessible.
import requests

FUSEKI_BASE   = "http://localhost:3030"
FUSEKI_QUERY  = f"{FUSEKI_BASE}/datagovkg/sparql"
FUSEKI_UPDATE = f"{FUSEKI_BASE}/datagovkg/update"
FUSEKI_AUTH   = ("admin", "admin") # Uses admin credentials for management endpoints.

# Test 1: server liveness
try:
    response = requests.get(f"{FUSEKI_BASE}/$/ping", timeout=5)
    print(f"Fuseki ping     : {response.status_code} ({response.text.strip()})")
except requests.exceptions.RequestException as e:
    print(f"Fuseki ping     : FAILED ({e})")

# Test 2: dataset discovery via admin endpoint
try:
    response = requests.get(
        f"{FUSEKI_BASE}/$/datasets",
        auth=FUSEKI_AUTH,
        timeout=5
    )
    datasets = [ds["ds.name"] for ds in response.json().get("datasets", [])]
    print(f"Datasets found  : {datasets}")
except requests.exceptions.RequestException as e:
    print(f"Datasets        : FAILED ({e})")

# Test 3: SPARQL query endpoint
try:
    response = requests.post(
        FUSEKI_QUERY,
        data={"query": "SELECT * WHERE { ?s ?p ?o } LIMIT 1"},
        headers={"Accept": "application/sparql-results+json"},
        timeout=5
    )
    print(f"SPARQL endpoint : {response.status_code} (OK)")
except requests.exceptions.RequestException as e:
    print(f"SPARQL endpoint : FAILED ({e})")

Fuseki ping     : 200 (2026-05-21T02:58:16.070+00:00)
Datasets found  : ['/datagovkg']
SPARQL endpoint : 200 (OK)


In [18]:
# End-to-End RDF Pipeline Test: pipeline: insert a test triple and retrieve it
# Purpose: Demonstrate the complete workflow we'll use throughout: the project: create -> insert -> retrieve -> verify. 
import requests

FUSEKI_QUERY  = "http://localhost:3030/datagovkg/sparql"
FUSEKI_UPDATE = "http://localhost:3030/datagovkg/update"

test_triple_insert = """
INSERT DATA {
    <urn:test:datagovkg-verification> <urn:test:hasStatus> "operational" .
}
"""

test_triple_select = """
SELECT ?status WHERE {
    <urn:test:datagovkg-verification> <urn:test:hasStatus> ?status .
}
"""

# Insert
insert_response = requests.post(
    FUSEKI_UPDATE,
    data={"update": test_triple_insert},
    timeout=5
)
print(f"Insert status   : {insert_response.status_code}")

# Retrieve
select_response = requests.post(
    FUSEKI_QUERY,
    data={"query": test_triple_select},
    headers={"Accept": "application/sparql-results+json"},
    timeout=5
)

bindings = select_response.json()["results"]["bindings"]
if bindings:
    retrieved_status = bindings[0]["status"]["value"]
    print(f"Retrieved value : '{retrieved_status}'")
    print(f"Integrity check : {'PASS' if retrieved_status == 'operational' else 'FAIL'}")
else:
    print("Retrieval failed: no bindings returned")

Insert status   : 401
Retrieved value : 'operational'
Integrity check : PASS


In [19]:
# # Inspect all triples currently stored in the Fuseki datasetز
# A utility cell that can be re-run anytime to inspect the dataset.
# Useful for verifying inserts, debugging, and confirming data integrity.

import requests

FUSEKI_QUERY = "http://localhost:3030/datagovkg/sparql"

all_triples_query = """
SELECT ?subject ?predicate ?object
WHERE {
    ?subject ?predicate ?object .
}
ORDER BY ?subject ?predicate
"""

response = requests.post(
    FUSEKI_QUERY,
    data={"query": all_triples_query},
    headers={"Accept": "application/sparql-results+json"},
    timeout=5
)

bindings = response.json()["results"]["bindings"]
print(f"Triples in dataset: {len(bindings)}\n")

for i, binding in enumerate(bindings, start=1):
    subject   = binding["subject"]["value"]
    predicate = binding["predicate"]["value"]
    obj       = binding["object"]["value"]
    obj_type  = binding["object"]["type"]
    
    # Format object based on its RDF type for readable output
    if obj_type == "literal":
        obj_display = f'"{obj}"'
    elif obj_type == "uri":
        obj_display = f"<{obj}>"
    else:
        obj_display = obj
    
    print(f"[{i}] <{subject}>")
    print(f"    <{predicate}>")
    print(f"    {obj_display}\n")

Triples in dataset: 1

[1] <urn:test:datagovkg-verification>
    <urn:test:hasStatus>
    "operational"

